In [ ]:
import requests
import json
import os
import time
from datetime import datetime

# 1. Configuration & Setup
BASE_URL = "https://hacker-news.firebaseio.io/v0"
HEADERS = {"User-Agent": "TrendPulse/1.0"}

# Keywords for categorization
CATEGORIES = {
    "technology": ["AI", "software", "tech", "code", "computer", "data", "cloud", "API", "GPU", "LLM"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["NFL", "NBA", "FIFA", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "NASA", "genome"],
    "entertainment": ["movie", "film", "music", "Netflix", "game", "book", "show", "award", "streaming"]
}

def get_category(title):
    """Checks the title against keywords to assign a category."""
    title_lower = title.lower()
    for category, keywords in CATEGORIES.items():
        for word in keywords:
            if word.lower() in title_lower:
                return category
    return None # Returns None if no keywords match

def fetch_trending_data():
    all_collected_stories = []

    # Track how many stories we have per category (Limit: 25 each)
    category_counts = {cat: 0 for cat in CATEGORIES}

    print("Fetching top story IDs...")
    try:
        # Step 1: Get top 500 story IDs
        response = requests.get(f"{BASE_URL}/topstories.json", headers=HEADERS)
        response.raise_for_status()
        story_ids = response.json()[:500]
    except Exception as e:
        print(f"Failed to fetch IDs: {e}")
        return []

    # Step 2: Fetch details for each story
    for story_id in story_ids:
        # Stop if we have filled all categories (5 categories * 25 stories = 125 total)
        if all(count >= 25 for count in category_counts.values()):
            break

        try:
            item_url = f"{BASE_URL}/item/{story_id}.json"
            item_data = requests.get(item_url, headers=HEADERS).json()

            # Skip if the item isn't a story or doesn't have a title
            if not item_data or item_data.get('type') != 'story' or 'title' not in item_data:
                continue

            # Assign category
            category = get_category(item_data['title'])

            # Only save if it fits a category and we need more for that category
            if category and category_counts[category] < 25:
                # Extract and format the specific fields required
                story_record = {
                    "post_id": item_data.get("id"),
                    "title": item_data.get("title"),
                    "category": category,
                    "score": item_data.get("score"),
                    "num_comments": item_data.get("descendants", 0),
                    "author": item_data.get("by"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                all_collected_stories.append(story_record)
                category_counts[category] += 1

                # Per instructions: Wait 2 seconds between each category loop
                # This logic checks if we just finished a category's quota
                if category_counts[category] == 25:
                    print(f"Finished collecting 25 stories for: {category}")
                    time.sleep(2)

        except Exception as e:
            print(f"Skipping story {story_id} due to error: {e}")
            continue

    return all_collected_stories

def save_to_json(data):
    """Creates the data folder and saves the stories to a JSON file."""
    if not os.path.exists('data'):
        os.makedirs('data')
        print("Created 'data/' directory.")

    # Format filename: trends_YYYYMMDD.json
    filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

    print(f"Collected {len(data)} stories. Saved to {filename}")

# Run the script
if __name__ == "__main__":
    collected_data = fetch_trending_data()
    if collected_data:
        save_to_json(collected_data)
    else:
        print("No stories were collected.")


Fetching top story IDs...
Failed to fetch IDs: HTTPSConnectionPool(host='hacker-news.firebaseio.io', port=443): Max retries exceeded with url: /v0/topstories.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7accccd829c0>: Failed to resolve 'hacker-news.firebaseio.io' ([Errno -2] Name or service not known)"))
No stories were collected.
